# 1단계: main `0dd7a35` 그대로 화장품 300장 평가

- **목적:** 선우 요청. main `0dd7a35`를 한 줄도 고치지 않고 화장품 900001~900300에 돌려 최초 성능을 기록한다.
- **데이터 성격:** main 코드는 이 300장을 보고 만든 규칙이 없다(main 기준 처음 보는 데이터). 팀 기준으로는 개발 데이터라 "외부 테스트셋"이라고 쓰지 않는다.
- **공식 결과:** `ocr_pipeline.predict_image_detailed`를 그대로 부른다. 엔진을 감싸 입력 해시·결과·시간만 기록한다.
- **분석용 기록:** 공식 경로가 안 돌린 단계도 같은 방법으로 돌려 5단계를 모두 저장한다(나중에 OCR 없이 재계산용).
- **확인 장치:** 제출 코드 해시, 라벨 해시(900259 수정본), 사진 300장 해시, 모델 해시를 기록한다. 검증용 900301~900400은 코드에서 막는다.

**실행:** 1 → 2 → 3 → 4 → 5번 셀 순서. 중간에 멈추면 4번만 다시(끝난 사진은 건너뜀). 커널을 재시작했으면 1 → 2 → 3 → 4.
**전원 연결 필수.** 시간은 전원이 연결된 사진만 집계한다. 예상 약 1.5~3시간(사진당 5단계).
**끝나면** `cos300_0dd7a35` 폴더를 통째로 압축해서 Claude에게 보낸다.

In [ ]:
# 1) 설정: 경로만 확인/수정
from pathlib import Path
CODE_DIR  = r"C:\Users\user\itda_OCR_0dd7a35"        # 이 노트북이 들어 있는 0dd7a35 폴더
DATA_DIR  = r"C:\Users\user\itda_OCR"                 # 기존 폴더 (사진·라벨·모델)
COS_DIR   = DATA_DIR + r"\Cosmetics"
LABEL_CSV = DATA_DIR + r"\cosmetics_labels_900001-900300.csv"
WEIGHTS   = DATA_DIR + r"\weights\paddleocr"
OUT_DIR   = CODE_DIR + r"\cos300_0dd7a35"
ENGINE_KW = dict(enable_mkldnn=True, cpu_threads=4, recognition_batch_size=6)   # initialize_engine 기본값

In [ ]:
# 2) 확인: 코드가 0dd7a35 그대로인지, 라벨·사진·모델 해시 기록 (OCR 없음)
import sys, json, time, platform, hashlib, statistics
if "ocr_pipeline" in sys.modules and not str(Path(sys.modules["ocr_pipeline"].__file__)).startswith(CODE_DIR):
    raise SystemExit("다른 폴더의 ocr_pipeline이 이미 불러와져 있습니다. 커널 재시작 후 1번부터 다시 실행하세요.")
sys.path.insert(0, CODE_DIR)
sys.path.insert(0, str(Path(CODE_DIR) / "eval_0dd7a35"))
import ocr_pipeline as op
import cos300_eval as ce
print("ocr_pipeline:", op.__file__)
man = ce.check_code(CODE_DIR)
print("코드 커밋:", man["commit"], "(제출 코드 파일", len(man["files"]), "개 해시 일치)")
LABELS, FILES = ce.check_inputs(COS_DIR, LABEL_CSV)
print("라벨 sha256:", ce.EXPECTED_LABEL_SHA[:16], "… 일치, 300줄")
OUT = Path(OUT_DIR); OUT.mkdir(parents=True, exist_ok=True)
t = time.perf_counter()
photos = {i: ce.sha256_file(FILES[i]) for i in ce.IDS}
models = {str(p.relative_to(WEIGHTS)): ce.sha256_file(p) for p in sorted(Path(WEIGHTS).rglob("*")) if p.is_file()}
print(f"사진 300장·모델 파일 {len(models)}개 해시 계산 {time.perf_counter()-t:.0f}초")
manifest = {"code": man, "label_csv": LABEL_CSV, "label_sha256": ce.EXPECTED_LABEL_SHA,
            "photos_sha256": photos, "models_sha256": models, "engine_kwargs": ENGINE_KW,
            "python": sys.version, "platform": platform.platform(), "processor": platform.processor()}
MAN = OUT / "run_manifest.json"
if MAN.exists():
    old = json.loads(MAN.read_text(encoding="utf-8"))
    diff = [k for k in ("code", "label_sha256", "photos_sha256", "models_sha256", "engine_kwargs") if old.get(k) != manifest[k]]
    if diff:
        raise SystemExit("이전 실행과 입력이 다릅니다: " + ", ".join(diff) + " — 멈추고 Claude에게 알려 주세요.")
    manifest["runs"] = old.get("runs", [])
else:
    manifest["runs"] = []
MAN.write_text(json.dumps(manifest, ensure_ascii=False, indent=1), encoding="utf-8")
print("전원 연결:", {True: "예", False: "아니오(배터리) — 전원을 연결하세요", None: "알 수 없음"}[ce.on_ac_power()])

In [ ]:
# 3) 모델 준비 + 예열 1장 (기록 안 함)
t0 = time.perf_counter()
ENGINE = op.initialize_engine(weights_dir=WEIGHTS, **ENGINE_KW)
print(f"모델 준비 {time.perf_counter()-t0:.1f}초")
t0 = time.perf_counter()
op.predict_image_detailed(ENGINE, FILES[ce.IDS[0]])
print(f"예열 1장 {time.perf_counter()-t0:.1f}초")

In [ ]:
# 4) 본 실행: 사진마다 공식 경로 + 나머지 단계. 멈추면 이 셀만 다시 실행.
OFF = OUT / "cos300_0dd7a35_official.jsonl"
ALL = OUT / "cos300_0dd7a35_allstages.jsonl"
done = {r["image_id"] for r in ce.read_jsonl(OFF)}
ce.read_jsonl(ALL)                                   # 끊긴 줄 정리
todo = [i for i in ce.IDS if i not in done]
print(f"남은 {len(todo)}장 / 300장", flush=True)
run = {"started": time.strftime("%Y-%m-%d %H:%M:%S"), "todo": len(todo), "ac_power_start": ce.on_ac_power()}
ce.keep_awake(True); started = time.perf_counter()
try:
    for k, i in enumerate(todo, 1):
        official, allstages = ce.process(ENGINE, i, FILES[i])
        official["label"] = LABELS[i]["final_date"]
        ce.append(ALL, allstages)                     # 분석용 먼저, 공식 기록은 마지막 (재시작 기준)
        ce.append(OFF, official)
        if k % 5 == 0 or k == len(todo):
            el = time.perf_counter() - started
            print(f"[{k}/{len(todo)}] {i} 경과 {el/60:.0f}분, 남은 예상 {el/k*(len(todo)-k)/60:.0f}분"
                  + ("" if official["ac_power"] is not False else "  ※ 배터리"), flush=True)
finally:
    ce.keep_awake(False)
    run["ended"] = time.strftime("%Y-%m-%d %H:%M:%S"); run["ac_power_end"] = ce.on_ac_power()
    m = json.loads(MAN.read_text(encoding="utf-8")); m["runs"].append(run)
    MAN.write_text(json.dumps(m, ensure_ascii=False, indent=1), encoding="utf-8")
print("끝")

In [ ]:
# 5) 재계산 일치 확인 + 미리 정한 지표 (OCR 없음)
import csv, math, collections
off = {r["image_id"]: r for r in ce.read_jsonl(OFF)}
alls = {r["image_id"]: r for r in ce.read_jsonl(ALL)}
assert sorted(off) == ce.IDS and sorted(alls) == ce.IDS, f"아직 끝나지 않았습니다: {len(off)}/300"
mismatch = []
for i in ce.IDS:
    p, m, a = ce.replay(alls[i])
    o = off[i]
    if (p, m, a) != (o["prediction"], o["method"], o["attempts"]):
        mismatch.append(i)
print("저장된 5단계로 다시 계산한 결과가 공식 실행과 같은가:", "300장 모두 같음" if not mismatch else f"다름 {mismatch}")

def wilson(k, n, z=1.96):
    p = k / n; c = (p + z*z/2/n) / (1 + z*z/n); h = z*math.sqrt(p*(1-p)/n + z*z/4/n/n) / (1 + z*z/n)
    return f"{100*p:.1f}% ({100*(c-h):.1f}–{100*(c+h):.1f})"
def final_stage(method):
    return "original_512" if method == "original_no_candidate" else max((s for s in ce.ALL_STAGES if method.startswith(s)), key=len)
def route(i):   # 평가용 경로 규칙 (6차 평가와 같음, 제출 코드 아님)
    o = off[i]; pred = o["prediction"]["final_date"]
    st = {s["stage"]: s for s in alls[i]["stages"]}[final_stage(o["method"])]
    _, ev = op.stage_result(st["detections"])
    from date_parser.select import find_all_candidates
    from date_parser.types import TextBox
    distinct = {c.result.final_date_string() for c in find_all_candidates([TextBox.from_dict(d) for d in st["detections"]])}
    if "NONE" in pred: return "MANUAL"
    if len(distinct) >= 2: return "CHOOSE"
    return "RECHECK" if (ev is None or ev["q"] is None or ev["q"] < 0.90) else "CONFIRM"
def late(pred, truth):   # 실제보다 늦은 날짜: 둘 다 있는 앞자리(연, 연월, 연월일)끼리 비교해 예측이 늦음
    if pred == "NONE" or truth == "NONE": return False
    p, t = pred.split("-"), truth.split("-"); n = 0
    while n < 3 and p[n] != "NONE" and t[n] != "NONE": n += 1
    return n > 0 and p[:n] > t[:n]

rows = []
for i in ce.IDS:
    o = off[i]; pred = o["prediction"]["final_date"]; truth = o["label"]
    base = op._run_baseline_cascade(lambda n, s={x["stage"]: x for x in alls[i]["stages"]}: op.stage_result(s[n]["detections"]))[0]["final_date"]
    rows.append({"image_id": i, "who": "수민" if int(i) <= 900150 else "서현", "label": truth, "prediction": pred,
                 "correct": int(pred == truth), "method": o["method"], "attempts": "+".join(o["attempts"]),
                 "route": route(i), "late": int(late(pred, truth)), "baseline_without_clahe1024": base,
                 "clahe1024_ran": int("clahe_1024" in o["attempts"]), "official_seconds": round(o["official_seconds"], 3),
                 "ac_power": o["ac_power"]})
with open(OUT / "cos300_0dd7a35_results.csv", "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

n = len(rows); ok = sum(r["correct"] for r in rows)
print(f"\n① 정답: {ok}/300 = {wilson(ok, n)}")
for who in ("수민", "서현"):
    R = [r for r in rows if r["who"] == who]; k = sum(r["correct"] for r in R)
    print(f"   {who} {k}/{len(R)} = {wilson(k, len(R))}")
none = sum(r["prediction"] == "NONE" for r in rows); part = sum("NONE" in r["prediction"] and r["prediction"] != "NONE" for r in rows)
print(f"② NONE {none}장 ({100*none/n:.1f}%), 부분 날짜 {part}장 ({100*part/n:.1f}%)")
rc = collections.Counter(r["route"] for r in rows)
print("③ 경로:", dict(rc), f"/ 직원 검토 비율 {wilson(n - rc['CONFIRM'], n)}")
cw = [r["image_id"] for r in rows if r["route"] == "CONFIRM" and not r["correct"]]
lt = [r for r in rows if r["late"]]
print(f"④ 확인 후 승인인데 틀림 {len(cw)}장 {cw}")
print(f"   늦은 날짜 오류 {len(lt)}장 {[r['image_id'] for r in lt]}, 그중 확인 후 승인 {sum(r['route']=='CONFIRM' for r in lt)}")
ran = [r for r in rows if r["clahe1024_ran"]]; changed = [r for r in ran if r["baseline_without_clahe1024"] != r["prediction"]]
print(f"⑤ clahe_1024 실행 {len(ran)}장, 결과가 바뀐 사진 {len(changed)}장:",
      [(r["image_id"], r["baseline_without_clahe1024"], "→", r["prediction"], "정답" if r["correct"] else "오답") for r in changed])
ac = [r["official_seconds"] for r in rows if r["ac_power"] is True]
if ac:
    q = statistics.quantiles(ac, n=10)
    print(f"⑥ 시간(전원 연결 {len(ac)}장): 중앙값 {statistics.median(ac):.2f}초, 평균 {statistics.mean(ac):.2f}초, p90 {q[8]:.2f}초")
else:
    print("⑥ 시간: 전원 연결 상태로 잰 사진이 없어 집계하지 않음")
st = collections.defaultdict(list)
for i in ce.IDS:
    if off[i]["ac_power"] is True:
        for s in alls[i]["stages"]: st[s["stage"]].append(s["seconds"])
print("   단계별 중앙값:", {k: round(statistics.median(v), 2) for k, v in st.items()})
print("\n보낼 것:", OUT)